# Lean-15 : Le Theoreme de Kochen-Specker (Cabello 18 vecteurs)

**Navigation** : [<< Lean-14 Conway Tribute](Lean-14-Conway-Tribute.ipynb) | [Index](README.md)

**Kernel** : Python 3 (illustrations + verifications combinatoires) + Lean 4 (WSL) pour la section 6

---

## Introduction

En 1967, Simon Kochen et Ernst Specker publient un theoreme qui exclut une large classe d'interpretations de la mecanique quantique : les **theories a variables cachees non-contextuelles**. La preuve originale utilise **117 vecteurs** de R^3. Plusieurs raffinements suivent (33 vecteurs par Peres 1991, 20 par Kernaghan 1994) jusqu'a la preuve la plus compacte connue avec **18 vecteurs de R^4** par Cabello, Estebaranz et Garcia-Alcaine en 1996. C'est cette derniere qui sert de noyau combinatoire au theoreme de libre arbitre de Conway-Kochen (2006).

Ce notebook est le **Pilier 1** de l'Epic #1651 (Theoreme de Libre Arbitre Conway-Kochen). Il presente :
1. La question : peut-on attribuer une valeur 0/1 aux observables quantiques de maniere non-contextuelle ?
2. La structure : 18 vecteurs de R^4 en 9 bases orthogonales, chacun appartenant a exactement 2 bases
3. L'argument de parite : 9 (impair) vs 2k (pair) -- contradiction
4. Verification computationnelle : aucune coloration valide n'existe sur 2^18 essais
5. Le port Lean 4 (Pilier 1 du theoreme de libre arbitre)
6. Verification `lake build` du scaffold
7. Pont avec le theoreme de libre arbitre de Conway-Kochen (Piliers 2-3)

### Prerequis
- Notions d'algebre lineaire (R^4, bases orthogonales, produit scalaire)
- Notebooks Lean-1 a Lean-7 pour les aspects formels (recommande)
- Lean-12 pour le pattern Lake build + WSL

### Duree estimee : 75 minutes

## 1. La Question de Kochen-Specker

### 1.1 Le contexte physique

En mecanique quantique, mesurer une grandeur revient a projeter l'etat du systeme sur une famille de vecteurs orthonormaux. Pour un systeme de spin-1, mesurer le carre du spin selon trois axes mutuellement orthogonaux (en 3D) ou quatre axes mutuellement orthogonaux (en 4D pour les paires entrelacees) donne toujours un seul resultat 1 et le reste 0.

**Variable cachee non-contextuelle (NCHV)** : une theorie qui attribue a chaque observable une valeur predeterminee, **independante du contexte de mesure** (i.e. des autres observables mesures simultanement).

Pour les observables associes a des directions vectorielles, cela revient a une fonction $f : \text{vecteurs} \to \{0, 1\}$ telle que dans **toute base orthogonale**, exactement un vecteur recoit la valeur 1.

### 1.2 Le theoreme

**Theoreme (Kochen-Specker 1967, version Cabello et al. 1996)** : Il existe un ensemble fini de vecteurs de R^4 -- precisement 18 vecteurs organises en 9 bases orthogonales -- tel qu'**aucune** fonction $f$ a valeurs dans $\{0, 1\}$ ne peut satisfaire la contrainte "un seul 1 par base".

**Consequence** : aucune theorie a variables cachees non-contextuelle ne peut reproduire les predictions de la mecanique quantique. Toute attribution de valeurs predeterminees doit **dependre du contexte** ou abandonner la realisme local.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from itertools import product
from collections import Counter

np.set_printoptions(suppress=True, precision=4)
print('Setup ok : numpy', np.__version__)

Setup ok : numpy 2.3.4


## 2. Les 18 Vecteurs de Cabello

Les 18 vecteurs de la preuve Cabello et al. (1996) sont enumeres ci-dessous. Ils sont initialement donnes a normalisation pres -- les coordonnees brutes (entieres) suffisent pour les arguments combinatoires d'orthogonalite (le produit scalaire vaut 0 si et seulement si la version normalisee est orthogonale).

**Source** : Cabello, Estebaranz, Garcia-Alcaine, "Bell-Kochen-Specker theorem: A proof with 18 vectors", Phys. Lett. A 212 (1996), 183-187. Table reproduite dans Wikipedia, section Overview.

In [2]:
# Les 18 vecteurs distincts (coordonnees brutes, R^4)
vectors = {
    0:  ( 0,  0,  0,  1),  # v0
    1:  ( 0,  0,  1,  0),  # v1
    2:  ( 1,  1,  0,  0),  # v2
    3:  ( 1, -1,  0,  0),  # v3
    4:  ( 0,  1,  0,  0),  # v4
    5:  ( 1,  0,  1,  0),  # v5
    6:  ( 1,  0, -1,  0),  # v6
    7:  ( 1, -1,  1, -1),  # v7
    8:  ( 1, -1, -1,  1),  # v8
    9:  ( 0,  0,  1,  1),  # v9
    10: ( 1,  1,  1,  1),  # v10
    11: ( 0,  1,  0, -1),  # v11
    12: ( 1,  0,  0,  1),  # v12
    13: ( 1,  0,  0, -1),  # v13
    14: ( 0,  1, -1,  0),  # v14
    15: ( 1,  1, -1,  1),  # v15
    16: ( 1,  1,  1, -1),  # v16
    17: (-1,  1,  1,  1),  # v17
}

# Verifier l'unicite (18 vecteurs distincts apres deduplication)
unique_vecs = {v: k for k, v in vectors.items()}
assert len(unique_vecs) == 18, f'Doublons detectes : {18 - len(unique_vecs)}'
print(f'18 vecteurs distincts : OK')
print(f'Norme L2 (au carre) par vecteur : {sorted(set(sum(x*x for x in v) for v in vectors.values()))}')

18 vecteurs distincts : OK
Norme L2 (au carre) par vecteur : [1, 2, 4]


### Interpretation : les 18 vecteurs

Les 18 vecteurs se groupent en 3 classes selon leur norme :

| Type | Exemple | Norme^2 | Nb vecteurs |
|------|---------|---------|-------------|
| Standards (1 coord) | (0,0,0,1) | 1 | 2 |
| Plans (2 coords +/-) | (1,1,0,0), (1,-1,0,0) | 2 | 8 |
| Quadruplets (4 coords +/-) | (1,1,1,1), (1,-1,1,-1) | 4 | 8 |

Tous deviennent unitaires apres normalisation. L'orthogonalite (produit scalaire = 0) n'est pas affectee par la normalisation.

## 3. Les 9 Bases Orthogonales

Les 18 vecteurs s'organisent en 9 bases orthogonales ("contextes") de 4 vecteurs chacune. Chaque base correspond a une experience de mesure simultanee compatible.

Notation : `Ck = (i, j, k, l)` signifie que la k-ieme base contient les vecteurs d'indices i, j, k, l (dans l'ordre du tableau Cabello original).

In [3]:
# Les 9 contextes (bases orthogonales) -- chaque entree liste 4 indices de vecteurs
contexts = [
    ( 0,  1,  2,  3),  # C0
    ( 0,  4,  5,  6),  # C1
    ( 7,  8,  2,  9),  # C2
    ( 7, 10,  6, 11),  # C3
    ( 1,  4, 12, 13),  # C4
    ( 8, 10, 13, 14),  # C5
    (15, 16,  3,  9),  # C6
    (15, 17,  5, 11),  # C7
    (16, 17, 12, 14),  # C8
]

def dot(u, v):
    return sum(a * b for a, b in zip(u, v))

# Verifier que chaque contexte forme 4 vecteurs mutuellement orthogonaux
ortho_ok = True
for k, ctx in enumerate(contexts):
    pairs = [(i, j) for idx_i, i in enumerate(ctx) for j in ctx[idx_i+1:]]
    for i, j in pairs:
        d = dot(vectors[i], vectors[j])
        if d != 0:
            print(f'  C{k} : v{i} . v{j} = {d} (non-orthogonal !)')
            ortho_ok = False
    if ortho_ok:
        pass

print(f'\n9 contextes, 6 paires par contexte = 54 paires verifiees')
print(f'Toutes orthogonales : {ortho_ok}')


9 contextes, 6 paires par contexte = 54 paires verifiees
Toutes orthogonales : True


### Interpretation : verification d'orthogonalite

Chaque base C0-C8 contient 4 vecteurs mutuellement orthogonaux. Cela donne 6 paires par base (C(4,2) = 6) et 54 paires au total a verifier. La verification renvoie `True` : la table Cabello respecte effectivement la contrainte geometrique d'orthogonalite, prerequis pour interpreter chaque base comme une mesure quantique compatible.

## 4. La Structure d'Overlap (chaque vecteur dans 2 bases)

La cle combinatoire du theoreme est que les 18 vecteurs et les 9 bases sont relies par une **structure d'overlap** : chaque vecteur appartient a exactement 2 des 9 bases.

Pourquoi ? Une base a 4 vecteurs, donc 9 bases x 4 = 36 "slots". Si chaque vecteur etait dans une seule base, il faudrait 36 vecteurs distincts -- on n'en a que 18. Donc le ratio est 36/18 = 2 en moyenne. La table Cabello realise ce ratio **uniformement** : exactement 2 pour chaque vecteur, ni 1, ni 3.

In [4]:
# Compter les occurrences de chaque vecteur dans les 9 contextes
occurrences = Counter()
for ctx in contexts:
    for i in ctx:
        occurrences[i] += 1

# Verifier l'invariant : chaque vecteur apparait exactement 2 fois
print(f'{"Vec":<5} | {"#bases":<8} | {"Bases contenant ce vecteur"}')
print('-' * 65)
all_two = True
for v in range(18):
    bases = [k for k, ctx in enumerate(contexts) if v in ctx]
    print(f'v{v:<4} | {occurrences[v]:<8} | C{bases[0]} et C{bases[1]}' if len(bases)==2 else f'v{v:<4} | {occurrences[v]:<8} | {bases} (PROBLEME)')
    if occurrences[v] != 2:
        all_two = False

print(f'\nChaque vecteur dans exactement 2 bases : {all_two}')
print(f'Total slots = 36 = 9 bases x 4 = 18 vecteurs x 2 : {sum(occurrences.values()) == 36}')

Vec   | #bases   | Bases contenant ce vecteur
-----------------------------------------------------------------
v0    | 2        | C0 et C1
v1    | 2        | C0 et C4
v2    | 2        | C0 et C2
v3    | 2        | C0 et C6
v4    | 2        | C1 et C4
v5    | 2        | C1 et C7
v6    | 2        | C1 et C3
v7    | 2        | C2 et C3
v8    | 2        | C2 et C5
v9    | 2        | C2 et C6
v10   | 2        | C3 et C5
v11   | 2        | C3 et C7
v12   | 2        | C4 et C8
v13   | 2        | C4 et C5
v14   | 2        | C5 et C8
v15   | 2        | C6 et C7
v16   | 2        | C6 et C8
v17   | 2        | C7 et C8

Chaque vecteur dans exactement 2 bases : True
Total slots = 36 = 9 bases x 4 = 18 vecteurs x 2 : True


### Interpretation : invariant combinatoire verifie

Le tableau confirme que **les 18 vecteurs apparaissent chacun dans exactement 2 des 9 bases**. C'est cette invariance qui rend l'argument de parite possible -- si la repartition etait inegale (par exemple un vecteur dans 3 bases ou un autre dans 1 seule), l'argument echouerait et il pourrait exister une coloration valide.

Cette propriete est exactement ce que la lemme Lean `each_vector_in_two_contexts` enonce dans `KochenSpecker.lean` (cf section 5).

## 5. L'Argument de Parite

Une coloration $c : \text{vecteurs} \to \{0, 1\}$ est dite **valide** si dans chaque base, exactement un vecteur recoit la valeur 1.

### 5.1 Le decompte global

Si une telle coloration existe, le **nombre total de 1 (compte avec multiplicite sur les bases)** est :
$$\sum_{k=0}^{8} (\text{ones dans } C_k) = 9 \cdot 1 = 9$$

puisque chaque base contribue exactement 1.

### 5.2 Le decompte par vecteur

Reordonnons la somme en regroupant par vecteur. Chaque vecteur $v$ contribue $c(v) \cdot (\text{nb de bases contenant } v)$. Avec la propriete d'overlap (chaque vecteur dans 2 bases) :
$$\sum_{k=0}^{8} (\text{ones dans } C_k) = \sum_{v=0}^{17} c(v) \cdot 2 = 2 \cdot \sum_{v=0}^{17} c(v)$$

Ce nombre est **pair**.

### 5.3 La contradiction

Une meme somme vaut a la fois 9 (impair) et 2k (pair). **Contradiction**. Donc aucune coloration valide n'existe.

Verifions ce raisonnement par recherche exhaustive sur les $2^{18} = 262\,144$ colorations possibles.

In [5]:
# Recherche exhaustive : enumerer les 2^18 colorations, compter combien sont valides
n_valid = 0
n_total = 2 ** 18
examples_per_count = {}

for bits in range(n_total):
    c = [(bits >> v) & 1 for v in range(18)]
    # Compter, pour chaque contexte, le nombre de 1
    sums = [sum(c[i] for i in ctx) for ctx in contexts]
    if all(s == 1 for s in sums):
        n_valid += 1
    # Distribution des nombres de contextes "OK" (= sum exactly 1)
    k_ok = sum(1 for s in sums if s == 1)
    examples_per_count[k_ok] = examples_per_count.get(k_ok, 0) + 1

print(f'Total colorations explorees : {n_total:,}')
print(f'Colorations valides (9/9 contextes OK) : {n_valid}')
print()
print(f'Distribution du nombre de contextes "exactement 1" :')
for k in sorted(examples_per_count.keys()):
    pct = 100.0 * examples_per_count[k] / n_total
    bar = '#' * int(pct / 2)
    print(f'  {k}/9 OK : {examples_per_count[k]:>7,} colorations ({pct:5.2f}%) {bar}')

Total colorations explorees : 262,144
Colorations valides (9/9 contextes OK) : 0

Distribution du nombre de contextes "exactement 1" :
  0/9 OK :  30,550 colorations (11.65%) #####
  1/9 OK :  59,760 colorations (22.80%) ###########
  2/9 OK :  66,780 colorations (25.47%) ############
  3/9 OK :  51,768 colorations (19.75%) #########
  4/9 OK :  33,264 colorations (12.69%) ######
  5/9 OK :  13,248 colorations ( 5.05%) ##
  6/9 OK :   5,748 colorations ( 2.19%) #
  7/9 OK :     792 colorations ( 0.30%) 
  8/9 OK :     234 colorations ( 0.09%) 


### Interpretation : verification computationnelle de Kochen-Specker

La recherche exhaustive confirme : **aucune coloration parmi les 262 144 ne satisfait simultanement les 9 contraintes**. C'est la verification "par force brute" du theoreme de Kochen-Specker pour la table Cabello a 18 vecteurs.

On peut aussi observer la distribution : un nombre significatif de colorations satisfait 7 ou 8 contextes sur 9, mais aucune ne franchit la barre de 9/9 -- ce qui aligne avec l'intuition que la contradiction est globale, pas locale.

**Note pedagogique** : la preuve formelle Lean (theorem `kochen_specker` dans `KochenSpecker.lean`) ne procede pas par enumeration mais par argument symbolique (parite), ce qui generalise immediatement a tout n'importe quel ensemble similaire et donne une preuve courte (~10 lignes apres les lemmes prealables).

## 6. Le Port Lean 4 : `KochenSpecker.lean` (Pilier 1)

Le fichier `conway_lean/Conway/KochenSpecker.lean` formalise le theoreme de Kochen-Specker pour la table Cabello. Il sert de **Pilier 1** dans la construction du theoreme de libre arbitre de Conway-Kochen (Epic #1651).

### 6.1 Strategie : formulation abstraite

Plutot que de formaliser les coordonnees R^4 et la verification d'orthogonalite (qui requiert des produits scalaires reels), le scaffold encode la structure combinatoire :
- 18 indices de vecteurs : `VecIdx := Fin 18`
- 9 indices de contextes : `ContextIdx := Fin 9`
- La fonction `contextMembers : ContextIdx -> Fin 4 -> VecIdx` qui realise la table Cabello

L'orthogonalite de chaque contexte est implicite : verifier les produits scalaires est une charge separee, reportee a Pilier 2 (FreeWillTheorem.lean).

### 6.2 La definition `contextMembers`

```lean
/-- An abstract index for the 18 distinct vectors. -/
abbrev VecIdx := Fin 18

/-- An abstract index for the 9 orthogonal bases (contexts). -/
abbrev ContextIdx := Fin 9

/-- Context membership: which 4 vector indices form each orthogonal basis. -/
def contextMembers : ContextIdx -> Fin 4 -> VecIdx
  -- C0: {v0, v1, v2, v3}
  | 0, 0 => 0 | 0, 1 => 1 | 0, 2 => 2 | 0, 3 => 3
  -- C1: {v0, v4, v5, v6}
  | 1, 0 => 0 | 1, 1 => 4 | 1, 2 => 5 | 1, 3 => 6
  -- ... C2 a C8 sur le meme modele
```

Cette definition encode exactement la table verifiee numeriquement en section 3. Le pattern matching sur `(k : Fin 9, i : Fin 4)` couvre les 36 slots de la table.

### 6.3 La coloration valide et le lemme d'overlap

```lean
/-- A {0,1}-coloring of the 18 vectors. -/
def Coloring := VecIdx -> Bool

/-- A coloring is valid iff every context has exactly one vector colored true. -/
def IsValidColoring (c : Coloring) : Prop :=
  forall k : ContextIdx,
    (∑ i : Fin 4, if c (contextMembers k i) then (1 : ℕ) else 0) = 1

/-- Key property: each of the 18 vectors appears in exactly 2 contexts. -/
lemma each_vector_in_two_contexts (v : VecIdx) :
    (∑ k : ContextIdx, ∑ i : Fin 4,
      if contextMembers k i = v then (1 : ℕ) else 0) = 2 := by
  sorry  -- TODO Pilier 1: verify by `decide` / explicit case-split on v
```

Le lemme `each_vector_in_two_contexts` est l'enonce formel de la propriete combinatoire verifiee en section 4 (chaque vecteur dans exactement 2 bases).

### 6.4 Le theoreme principal et la preuve de parite

```lean
/-- **Kochen-Specker Theorem (18-vector Cabello proof)**. -/
theorem kochen_specker : ¬ ∃ c : Coloring, IsValidColoring c := by
  sorry  -- TODO Pilier 1: parity argument via Finset double-sum reorder
```

Le squelette de preuve (a fournir) suit exactement l'argument de la section 5 :

1. Supposer une coloration valide `c`. Sommer sur les 9 contextes : ∑_k (somme dans C_k) = 9 (par `IsValidColoring`).
2. Reordonner la double somme via `Finset.sum_comm` :
   ∑_k ∑_i [c(contextMembers k i)] = ∑_v c(v) · (nb de contextes contenant v) = ∑_v c(v) · 2 (par `each_vector_in_two_contexts`)
3. Donc 9 = 2 · (∑_v c(v)), ce qui contredit la parite (9 est impair, 2k pair).

**Statut du scaffold** : 2 sorrys (les 2 a remplir lors du remplissage Pilier 1).

## 7. Verification : `lake build` + grep sorry

Verifions que le scaffold compile (modulo les 2 sorrys attendus) et que le nombre de sorrys est exactement celui annonce.

In [6]:
import subprocess

lean_project = '/mnt/c/dev/CoursIA/MyIA.AI.Notebooks/SymbolicAI/Lean/conway_lean'
build_timeout_s = 900  # 15 min — generous pour fresh-cache scenarios (mathlib pull)

print('Verification du scaffold KochenSpecker.lean (lake build via WSL)...')
print('-' * 60)

try:
    result = subprocess.run(
        ['wsl', 'bash', '-c',
         f'cd {lean_project} && source ~/.elan/env 2>/dev/null; lake build Conway.KochenSpecker 2>&1 | tail -15'],
        capture_output=True, text=True, timeout=build_timeout_s
    )
    output = result.stdout.strip()
    print(output[-1800:] if len(output) > 1800 else output)
    if result.stderr:
        print('STDERR:', result.stderr[-300:])
    print(f'\nExit code : {result.returncode}')
    print('0 = SUCCESS (compile avec sorry warning), autre = ECHEC')
except subprocess.TimeoutExpired:
    print(f'TimeoutExpired apres {build_timeout_s}s (cache mathlib pas prechauffe sur cette machine).')
    print('La verification CI / build local du PR #1656 reste autoritative.')
    print('Pour relancer manuellement :')
    print(f'  wsl bash -c "cd {lean_project} && lake build Conway.KochenSpecker"')


Verification du scaffold KochenSpecker.lean (lake build via WSL)...
------------------------------------------------------------


⚠ [2946/2946] Replayed Conway.KochenSpecker
Build completed successfully (2946 jobs).

Exit code : 0
0 = SUCCESS (compile avec sorry warning), autre = ECHEC


In [7]:
# Compter les sorrys dans le scaffold KochenSpecker.lean
result = subprocess.run(
    ['wsl', 'bash', '-c',
     f'cd {lean_project} && grep -c sorry Conway/KochenSpecker.lean'],
    capture_output=True, text=True, timeout=10
)

count = result.stdout.strip()
print(f'Nombre de sorrys dans Conway/KochenSpecker.lean : {count}')
print(f'Attendu (Pilier 1 scaffold) : 2')
print(f'Match : {count == "2"}')

# Localiser les sorrys
result2 = subprocess.run(
    ['wsl', 'bash', '-c',
     f'cd {lean_project} && grep -n sorry Conway/KochenSpecker.lean'],
    capture_output=True, text=True, timeout=10
)
print(f'\nLocalisation :')
print(result2.stdout.strip())

Nombre de sorrys dans Conway/KochenSpecker.lean : 2
Attendu (Pilier 1 scaffold) : 2
Match : True

Localisation :
159:  sorry  -- TODO Pilier 1: verify by `decide` / explicit case-split on v
176:  sorry  -- TODO Pilier 1: parity argument via Finset double-sum reorder


### Interpretation : statut du Pilier 1

| Verification | Resultat attendu | Signification |
|--------------|------------------|---------------|
| `lake build Conway.KochenSpecker` | Exit code 0 + 2 warnings sorry | Le scaffold compile, les types sont coherents |
| Nombre de sorrys | 2 | `each_vector_in_two_contexts` + `kochen_specker` (TODO Pilier 1) |
| Localisation des sorrys | Bloc TODO Pilier 1 | Coherent avec la documentation |

Une fois les 2 sorrys remplis (via `decide` pour le lemme + argument de parite Finset pour le theoreme), le module `Conway/KochenSpecker.lean` realisera entierement Pilier 1 de l'Epic #1651.

## 8. Pont avec le Theoreme de Libre Arbitre de Conway-Kochen

Le theoreme de Kochen-Specker est le **noyau combinatoire** du theoreme de libre arbitre (Free Will Theorem) de Conway et Kochen (2006, 2009). Ce theoreme affirme :

> Si la reponse d'un experimentateur a un choix de mesure n'est pas une fonction de tout ce qui s'est passe avant, alors la reponse de la particule mesuree non plus.

Plus precisement, le theoreme repose sur 3 axiomes (Piliers 2 du port Lean) :

| Axiome | Enonce intuitif | Pilier Lean |
|--------|-----------------|--------------|
| **SPIN** | Pour les particules de spin 1, mesurer le carre du spin selon 3 axes orthogonaux donne toujours (1, 1, 0) dans un certain ordre | Utilise KS (Pilier 1) |
| **TWIN** | Deux particules de spin 1 peuvent etre entrelacees de sorte que leurs mesures concordent toujours | Axiome dans FreeWillTheorem.lean |
| **MIN** | Les choix d'experimentateurs (parametres de mesure) ne sont pas determines par le passe causal | Axiome formel |

### 8.1 Architecture du port Lean

```
conway_lean/
  Conway/
    KochenSpecker.lean       <-- Pilier 1 (ce notebook)
    FreeWillTheorem.lean      <-- Pilier 2 (a venir : SPIN+TWIN+MIN -> contradiction)
    [Doomsday, LookAndSay,    <-- Autres modules Conway (Doomsday Algorithm, etc.)
     Fractran, Nim, Angel]
```

### 8.2 Etat actuel (mai 2026)

- **Pilier 1** : scaffold pousse via PR [#1656](https://github.com/jsboige/CoursIA/pull/1656), 2 sorrys (lemme overlap + theoreme parite). Verification combinatoire : ce notebook (Pilier 3 Lean-15).
- **Pilier 2** : a venir (Epic #1651). Axiomes SPIN+TWIN+MIN et theoreme principal.
- **Pilier 3** : notebook companion (ce fichier), Pilier educatif/illustratif.

## 9. Pour aller plus loin

### References historiques

1. **S. Kochen, E. P. Specker**, "The Problem of Hidden Variables in Quantum Mechanics", J. Math. Mech. 17 (1967), 59-87. La preuve originale a 117 vecteurs en R^3.

2. **A. Peres**, "Two simple proofs of the Kochen-Specker theorem", J. Phys. A 24 (1991), L175-L178. Preuve a 33 vecteurs.

3. **M. Kernaghan**, "Bell-Kochen-Specker theorem for 20 vectors", J. Phys. A 27 (1994), L829-L830.

4. **A. Cabello, J. M. Estebaranz, G. Garcia-Alcaine**, "Bell-Kochen-Specker theorem: A proof with 18 vectors", Phys. Lett. A 212 (1996), 183-187. **La table utilisee dans ce notebook**.

5. **J. H. Conway, S. Kochen**, "The Free Will Theorem", Found. Phys. 36 (2006), 1441-1473. arXiv:quant-ph/0604079.

6. **J. H. Conway, S. Kochen**, "The Strong Free Will Theorem", Notices AMS 56 (2009), 226-232.

### Implications philosophiques

Le theoreme KS et sa generalisation au theoreme de libre arbitre demolissent une classe entiere d'interpretations realistes locales de la mecanique quantique :

- **Bohm-De Broglie (pilot wave)** : sauve si on accepte la contextualite (les valeurs cachees dependent du contexte de mesure)
- **Many-Worlds (Everett)** : non concerne (pas de variables cachees)
- **GRW (collapse)** : non concerne
- **Local hidden variables** : exclu

### Liens vers d'autres notebooks

- [Lean-12 Sensitivity](Lean-12-Sensitivity-Theorem.ipynb) : autre theoreme combinatoire avec preuve Lean compacte (Huang 2019)
- [Lean-14 Conway Tribute](Lean-14-Conway-Tribute.ipynb) : hommage Conway, Game of Life
- [Lean-11 TorchLean](Lean-11-TorchLean.ipynb) : verification formelle pour ML

### Exercices

1. Verifier manuellement que la base C2 = (v7, v8, v2, v9) est orthogonale en calculant les 6 produits scalaires.
2. Pour la table de Peres a 33 vecteurs (R^3), construire un tableau similaire. Quelle est la structure d'overlap (chaque vecteur dans combien de triplets) ?
3. Lire `conway_lean/Conway/KochenSpecker.lean` et remplir le sorry du lemme `each_vector_in_two_contexts` avec `decide` ou un case-split explicite.
4. Coder l'argument de parite en Lean 4 (a partir du sketch en section 6.4) en utilisant `Finset.sum_comm`.

---

**Navigation** : [<< Lean-14 Conway Tribute](Lean-14-Conway-Tribute.ipynb) | [Index](README.md)